# Synthetic Data Analysis Notebook

Zero-setup notebook for distribution, validity, privacy, and visual evaluation checks.

1. Install dependencies
2. Set your data paths
3. Run analyzer
4. Generate visual report artifacts
5. Review tables and charts

In [ ]:
import sys
import subprocess

subprocess.check_call([sys.executable, "-m", "pip", "install", "--upgrade", "pip"])
subprocess.check_call([sys.executable, "-m", "pip", "install", "pandas", "numpy", "openpyxl", "pyarrow", "matplotlib", "seaborn"])

In [ ]:
from pathlib import Path

# TODO: Update these two paths before running.
DATA_PATH = Path("/path/to/your_real_level1_synthetic_data.csv")
METADATA_PATH = Path("/path/to/CAMCAN_Metadata_SynthSprint (1).xlsx")

cwd = Path.cwd()
_candidates = [
    cwd / "analyze_synthetic_data.py",
    cwd / "upload_to_virtual_workspace" / "analyze_synthetic_data.py",
]
ANALYZER_PATH = next((p for p in _candidates if p.exists()), None)
if ANALYZER_PATH is None:
    raise FileNotFoundError("Could not find analyze_synthetic_data.py. Run notebook from upload folder.")

OUTDIR = cwd / "output"
OUTDIR.mkdir(parents=True, exist_ok=True)
print("Analyzer:", ANALYZER_PATH)
print("Output folder:", OUTDIR.resolve())

In [ ]:
import sys
import subprocess

cmd = [
    sys.executable,
    str(ANALYZER_PATH),
    "--data", str(DATA_PATH),
    "--outdir", str(OUTDIR),
]
if METADATA_PATH.exists():
    cmd.extend(["--metadata", str(METADATA_PATH)])
else:
    print("Metadata file not found; running without --metadata")

print("Running:", " ".join(cmd))
subprocess.check_call(cmd)

VIS_SCRIPT = ANALYZER_PATH.parent / "visualize_synthetic_report.py"
if not VIS_SCRIPT.exists():
    raise FileNotFoundError(f"Missing visualization script: {VIS_SCRIPT}")

viz_cmd = [sys.executable, str(VIS_SCRIPT), "--outdir", str(OUTDIR)]
print("Running:", " ".join(viz_cmd))
subprocess.check_call(viz_cmd)

In [ ]:
import json
import pandas as pd

summary = json.loads((OUTDIR / "summary.json").read_text(encoding="utf-8"))
summary

In [ ]:
import pandas as pd

def load_if_exists(filename):
    p = OUTDIR / filename
    return pd.read_csv(p) if p.exists() else None

column_profile = load_if_exists("column_profile.csv")
privacy_summary = load_if_exists("privacy_risk_summary.csv")
pii_hits = load_if_exists("pii_value_pattern_hits.csv")
top_corr = load_if_exists("numeric_top_correlations.csv")

display(column_profile.head(20) if column_profile is not None else "column_profile.csv not found")
display(privacy_summary if privacy_summary is not None else "privacy_risk_summary.csv not found")
display(pii_hits if pii_hits is not None else "pii_value_pattern_hits.csv not found")
display(top_corr.head(20) if top_corr is not None else "numeric_top_correlations.csv not found")

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

VIS_DIR = OUTDIR / "visuals"
if not VIS_DIR.exists():
    print("No visuals directory found.")
else:
    image_paths = sorted(VIS_DIR.glob("*.png"))
    if not image_paths:
        print("No PNG visuals found.")
    else:
        print("Visual files:")
        for p in image_paths:
            print("-", p.name)

        for p in image_paths:
            img = mpimg.imread(p)
            plt.figure(figsize=(12, 7))
            plt.imshow(img)
            plt.axis("off")
            plt.title(p.name)
            plt.show()